In [1]:
# ============================================================
# Notebook 25
# 25_probability_fold_source_map
# Probability fold-source map and duplicate prediction handling
#
# Purpose:
#   Document how overlapping walk-forward probability predictions
#   are deduplicated before allocation.
#
# Outputs:
#   - outputs/AURORA_TWETF/diagnostics/probability_fold_source_map.csv
#   - outputs/AURORA_TWETF/tables/table_S5b_duplicate_probability_handling.csv
#   - outputs/AURORA_TWETF/reports/NOTEBOOK25_probability_fold_source_map_validation_report_*.json
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import re
import hashlib
import warnings

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive mount skipped or already mounted:", repr(e))

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. Paths and settings
# ------------------------------------------------------------

PROJECT_CODE = "AURORA_TWETF"
PROJECT_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / PROJECT_CODE

TABLE_DIR = OUTPUT_ROOT / "tables"
DIAG_DIR = OUTPUT_ROOT / "diagnostics"
REPORT_DIR = OUTPUT_ROOT / "reports"

for d in [TABLE_DIR, DIAG_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# This is the Notebook 07 run used by the AURORA pipeline.
NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"
NOTEBOOK08_INPUT_INDEX = NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"

STRICT_START = pd.Timestamp("2024-11-27")
STRICT_END = pd.Timestamp("2026-03-25")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

RUN_ROOT = OUTPUT_ROOT / "probability_fold_source_map" / f"run_{RUN_ID}"
RUN_TABLE_DIR = RUN_ROOT / "tables"
RUN_DIAG_DIR = RUN_ROOT / "diagnostics"
RUN_REPORT_DIR = RUN_ROOT / "reports"

for d in [RUN_ROOT, RUN_TABLE_DIR, RUN_DIAG_DIR, RUN_REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Notebook 25: probability fold-source map")
print("RUN_ID:", RUN_ID)
print("NOTEBOOK08_INPUT_INDEX:", NOTEBOOK08_INPUT_INDEX)
print("=" * 100)

if not NOTEBOOK08_INPUT_INDEX.exists():
    raise FileNotFoundError(f"Missing allocation input index: {NOTEBOOK08_INPUT_INDEX}")

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    path = Path(path)
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    # Flexible date parsing
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df[df["date"].notna()].copy()
        df = df.set_index("date")
    elif "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df[df["Date"].notna()].copy()
        df = df.set_index("Date")
    else:
        try:
            parsed = pd.to_datetime(df.index, errors="coerce")
            if pd.Series(parsed).notna().mean() > 0.5:
                df.index = parsed
        except Exception:
            pass

    df.index.name = "date"
    return df.sort_index()

def infer_horizon_from_target(target_col):
    s = str(target_col).lower()
    if "20" in s:
        return "20-day"
    if "60" in s:
        return "60-day"
    return "unknown"

def extract_fold_number(fold_id):
    m = re.search(r"(\d+)", str(fold_id))
    if m:
        return int(m.group(1))
    return 0

def build_fold_source_map(prob_df, horizon_label, split_filter=("validation", "test")):
    df = prob_df.copy()

    if "split" not in df.columns:
        df["split"] = "unknown"

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    # Keep only eligible out-of-sample rows by default.
    if split_filter is not None:
        df = df[df["split"].isin(list(split_filter))].copy()

    if df.empty:
        raise ValueError(f"No eligible probability rows for {horizon_label} after split filtering.")

    df = df.reset_index()
    if "date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "date"})

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()].copy()

    split_priority_map = {
        "train": 0,
        "training": 0,
        "validation": 1,
        "valid": 1,
        "val": 1,
        "test": 2,
        "oos": 2,
    }

    df["split_priority"] = df["split"].astype(str).str.lower().map(split_priority_map).fillna(0).astype(int)
    df["fold_number"] = df["fold_id"].map(extract_fold_number).astype(int)

    # This is the actual latest-eligible-fold rule.
    df_sorted = df.sort_values(["date", "split_priority", "fold_number"])

    # Count eligible rows per date before deduplication.
    counts = (
        df_sorted.groupby("date")
        .size()
        .rename("number_of_eligible_predictions")
        .reset_index()
    )

    selected = df_sorted.drop_duplicates(subset=["date"], keep="last").copy()

    selected = selected.merge(counts, on="date", how="left")

    selected["horizon"] = horizon_label
    selected["deduplication_rule"] = "latest eligible fold after sorting by date, split priority, and fold number"
    selected["has_multiple_eligible_predictions"] = selected["number_of_eligible_predictions"] > 1

    keep_cols = [
        "date",
        "horizon",
        "fold_id",
        "fold_number",
        "split",
        "split_priority",
        "number_of_eligible_predictions",
        "has_multiple_eligible_predictions",
        "deduplication_rule",
    ]

    # Add model metadata if available.
    for c in ["model_name", "model_family", "target_col", "run_id"]:
        if c in selected.columns:
            keep_cols.append(c)

    # Add probability columns if available.
    for c in [f"proba_class_{i}" for i in range(5)]:
        if c in selected.columns:
            keep_cols.append(c)

    selected = selected[keep_cols].copy()
    selected = selected.sort_values(["horizon", "date"])

    duplicate_summary = {
        "horizon": horizon_label,
        "eligible_rows_before_deduplication": int(len(df_sorted)),
        "unique_dates_after_deduplication": int(selected["date"].nunique()),
        "dates_with_multiple_eligible_predictions": int(selected["has_multiple_eligible_predictions"].sum()),
        "max_eligible_predictions_for_one_date": int(selected["number_of_eligible_predictions"].max()),
        "split_counts_selected": selected["split"].value_counts().to_dict(),
        "fold_counts_selected": selected["fold_id"].value_counts().to_dict(),
    }

    return selected, duplicate_summary

# ------------------------------------------------------------
# 2. Load allocation input index and probability files
# ------------------------------------------------------------

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

probability_file_rows = []

for _, row in input_index_df.iterrows():
    target_col = row.get("target_col", "")
    horizon = infer_horizon_from_target(target_col)

    p_parquet = Path(str(row.get("probability_path_parquet", "")))
    p_csv = Path(str(row.get("probability_path_csv", "")))

    if p_parquet.exists():
        p = p_parquet
    elif p_csv.exists():
        p = p_csv
    else:
        print("WARNING: probability file missing for row:", row.to_dict())
        continue

    probability_file_rows.append({
        "target_col": target_col,
        "horizon": horizon,
        "probability_path": p,
    })

if not probability_file_rows:
    raise FileNotFoundError("No probability files found from allocation input index.")

print("Probability files:")
for r in probability_file_rows:
    print(r["horizon"], r["target_col"], r["probability_path"])

# ------------------------------------------------------------
# 3. Build fold-source maps
# ------------------------------------------------------------

maps = []
summaries = []

for r in probability_file_rows:
    prob_df = read_table_auto(r["probability_path"])
    fold_map, summary = build_fold_source_map(
        prob_df=prob_df,
        horizon_label=r["horizon"],
        split_filter=("validation", "test"),
    )

    # Restrict to manuscript-aligned evaluation window for a separate flag.
    fold_map["in_aligned_evaluation_window"] = (
        (fold_map["date"] >= STRICT_START)
        & (fold_map["date"] <= STRICT_END)
    )

    fold_map["source_probability_file"] = str(r["probability_path"])
    fold_map["target_col_from_input_index"] = r["target_col"]

    maps.append(fold_map)
    summaries.append(summary)

fold_source_map = pd.concat(maps, ignore_index=True)
fold_source_map = fold_source_map.sort_values(["horizon", "date"])

summary_df = pd.DataFrame(summaries)

# Evaluation-window summary.
eval_summary = (
    fold_source_map[fold_source_map["in_aligned_evaluation_window"]]
    .groupby("horizon")
    .agg(
        selected_dates=("date", "nunique"),
        rows=("date", "size"),
        dates_with_multiple_eligible_predictions=("has_multiple_eligible_predictions", "sum"),
        max_eligible_predictions_for_one_date=("number_of_eligible_predictions", "max"),
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. Create Supplement Table S5b
# ------------------------------------------------------------

table_s5b = pd.DataFrame([
    {
        "item": "Duplicate source",
        "rule": "Overlapping walk-forward validation/test windows may generate more than one eligible out-of-sample probability row for the same calendar date.",
    },
    {
        "item": "Deduplication key",
        "rule": "Calendar date.",
    },
    {
        "item": "Sorting variables",
        "rule": "Date, split priority, and fold number.",
    },
    {
        "item": "Split priority",
        "rule": "Training < validation < test. Allocation inputs are drawn from eligible out-of-sample validation/test rows.",
    },
    {
        "item": "Retained row",
        "rule": "The final eligible row after sorting is retained for each calendar date.",
    },
    {
        "item": "Applied separately to",
        "rule": "20-day and 60-day probability files.",
    },
    {
        "item": "Final alignment",
        "rule": "Deduplicated 20-day and 60-day probability files are aligned on common dates before allocation.",
    },
    {
        "item": "Repository diagnostic",
        "rule": "outputs/AURORA_TWETF/diagnostics/probability_fold_source_map.csv",
    },
    {
        "item": "Claim use",
        "rule": "Defines the final probability sequence supplied to the allocation layer; does not use realized evaluation-period portfolio returns.",
    },
])

# ------------------------------------------------------------
# 5. Save outputs
# ------------------------------------------------------------

fold_source_map_path = RUN_DIAG_DIR / "probability_fold_source_map.csv"
summary_path = RUN_TABLE_DIR / "probability_fold_source_summary.csv"
eval_summary_path = RUN_TABLE_DIR / "probability_fold_source_evaluation_window_summary.csv"
table_s5b_path = RUN_TABLE_DIR / "table_S5b_duplicate_probability_handling.csv"

fold_source_map.to_csv(fold_source_map_path, index=False)
summary_df.to_csv(summary_path, index=False)
eval_summary.to_csv(eval_summary_path, index=False)
table_s5b.to_csv(table_s5b_path, index=False)

# Global copies.
global_fold_source_map_path = DIAG_DIR / "probability_fold_source_map.csv"
global_summary_path = TABLE_DIR / "probability_fold_source_summary.csv"
global_eval_summary_path = TABLE_DIR / "probability_fold_source_evaluation_window_summary.csv"
global_table_s5b_path = TABLE_DIR / "table_S5b_duplicate_probability_handling.csv"

fold_source_map.to_csv(global_fold_source_map_path, index=False)
summary_df.to_csv(global_summary_path, index=False)
eval_summary.to_csv(global_eval_summary_path, index=False)
table_s5b.to_csv(global_table_s5b_path, index=False)

# ------------------------------------------------------------
# 6. Validation report and manifest
# ------------------------------------------------------------

validation = {
    "project_code": PROJECT_CODE,
    "notebook": "25_probability_fold_source_map",
    "run_id": RUN_ID,
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "purpose": "Document duplicate probability handling for overlapping walk-forward predictions.",
    "allocation_input_index": str(NOTEBOOK08_INPUT_INDEX),
    "probability_files": [
        {
            "horizon": r["horizon"],
            "target_col": r["target_col"],
            "probability_path": str(r["probability_path"]),
        }
        for r in probability_file_rows
    ],
    "deduplication_rule": {
        "key": "calendar date",
        "sorting": ["date", "split_priority", "fold_number"],
        "retained_row": "final eligible row after sorting",
        "eligible_splits": ["validation", "test"],
        "split_priority": {"train": 0, "validation": 1, "test": 2},
        "applied_separately_to": ["20-day probability file", "60-day probability file"],
    },
    "summary": summaries,
    "evaluation_window_summary": eval_summary.to_dict(orient="records"),
    "outputs": {
        "fold_source_map": str(fold_source_map_path),
        "summary": str(summary_path),
        "evaluation_window_summary": str(eval_summary_path),
        "table_s5b": str(table_s5b_path),
        "global_fold_source_map": str(global_fold_source_map_path),
        "global_table_s5b": str(global_table_s5b_path),
    },
}

validation_path = RUN_REPORT_DIR / "NOTEBOOK25_probability_fold_source_map_validation_report.json"
global_validation_path = REPORT_DIR / f"NOTEBOOK25_probability_fold_source_map_validation_report_{RUN_ID}.json"

save_json(validation_path, validation)
save_json(global_validation_path, validation)

manifest_rows = []
for p in sorted(RUN_ROOT.rglob("*")):
    if p.is_file():
        stat = p.stat()
        manifest_rows.append({
            "path": p.relative_to(RUN_ROOT).as_posix(),
            "size_bytes": int(stat.st_size),
            "modified_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "sha256": sha256_file(p),
        })

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = RUN_REPORT_DIR / "NOTEBOOK25_file_manifest_SHA256.csv"
global_manifest_path = REPORT_DIR / f"NOTEBOOK25_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(global_manifest_path, index=False)

# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 25 COMPLETE")
print("=" * 100)
print("Run root:", RUN_ROOT)
print("Fold source map:", fold_source_map_path)
print("Global fold source map:", global_fold_source_map_path)
print("Table S5b:", table_s5b_path)
print("Global Table S5b:", global_table_s5b_path)
print("Validation report:", validation_path)
print("=" * 100)

print("\nOverall duplicate summary:")
display(summary_df)

print("\nAligned evaluation-window summary:")
display(eval_summary)

print("\nTable S5b:")
display(table_s5b)

print("\nFold-source map preview:")
display(fold_source_map.head(20))

Mounted at /content/drive
Notebook 25: probability fold-source map
RUN_ID: 20260725_005830
NOTEBOOK08_INPUT_INDEX: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv
Probability files:
20-day TAIEX_regime_fixed_20d /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selected_probabilities_TAIEX_regime_fixed_20d_E1_validation_weighted_probability_ensemble.parquet
60-day TAIEX_regime_fixed_60d /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/selected_probabilities_TAIEX_regime_fixed_60d_E1_validation_weighted_probability_ensemble.parquet

NOTEBOOK 25 COMPLETE
Run root: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/probability_fold_source_map/run_20260725_005830
Fold source map: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/probabil

,horizon,eligible_rows_before_deduplication,unique_dates_after_deduplication,dates_with_multiple_eligible_predictions,max_eligible_predictions_for_one_date,split_counts_selected,fold_counts_selected
0,20-day,1015,548,318,3,"{'test': 359, 'validation': 189}","{'WF1': 252, 'WF3': 170, 'WF2': 126}"
1,60-day,775,508,198,3,"{'test': 319, 'validation': 189}","{'WF1': 252, 'WF3': 130, 'WF2': 126}"



Aligned evaluation-window summary:


,horizon,selected_dates,rows,dates_with_multiple_eligible_predictions,max_eligible_predictions_for_one_date
0,20-day,319,319,235,3
1,60-day,319,319,195,3



Table S5b:


,item,rule
0,Duplicate source,Overlapping walk-forward validation/test windo...
1,Deduplication key,Calendar date.
2,Sorting variables,"Date, split priority, and fold number."
3,Split priority,Training < validation < test. Allocation input...
4,Retained row,The final eligible row after sorting is retain...
5,Applied separately to,20-day and 60-day probability files.
6,Final alignment,Deduplicated 20-day and 60-day probability fil...
7,Repository diagnostic,outputs/AURORA_TWETF/diagnostics/probability_f...
8,Claim use,Defines the final probability sequence supplie...



Fold-source map preview:


,date,horizon,fold_id,fold_number,split,split_priority,number_of_eligible_predictions,has_multiple_eligible_predictions,deduplication_rule,model_name,...,target_col,run_id,proba_class_0,proba_class_1,proba_class_2,proba_class_3,proba_class_4,in_aligned_evaluation_window,source_probability_file,target_col_from_input_index
0,2023-12-14,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.037495,0.211713,0.478908,0.264303,0.007580,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
1,2023-12-15,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.046152,0.230462,0.457935,0.260763,0.004687,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
2,2023-12-18,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.029283,0.255576,0.474890,0.234969,0.005282,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
3,2023-12-19,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.017047,0.256445,0.506755,0.216201,0.003552,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
4,2023-12-20,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.025995,0.231072,0.510132,0.226943,0.005859,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
5,2023-12-21,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.021417,0.266501,0.468376,0.236886,0.006820,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
6,2023-12-22,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.026989,0.248220,0.484646,0.232656,0.007488,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
7,2023-12-25,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.027226,0.239904,0.495908,0.233454,0.003509,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
8,2023-12-26,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.031999,0.248095,0.496206,0.219826,0.003875,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
9,2023-12-27,20-day,WF1,1,validation,1,1,False,"latest eligible fold after sorting by date, sp...",E1_validation_weighted_probability_ensemble,...,TAIEX_regime_fixed_20d,20260624_031817,0.033808,0.242498,0.494344,0.223062,0.006288,False,/content/drive/MyDrive/AURORA_TWETF/outputs/AU...,TAIEX_regime_fixed_20d
